# Carbon Intensity Forecast Model — XGBoost Multi-output Regressor Forecast

* This notebook uses the carbon intensity CSV file, applies pre-processing to the columns and trains an XGBoost multi-output regressor to predict a 24-hour forecast for carbon intensity in the UK national energy grid. The model predicts a 24-hr forecast as one output.

* The data is from NESO - 
The GB carbon intensity 𝐶𝑡 at time 𝑡 is found by weighting the carbon intensity 𝑐𝑔 for fuel type 𝑔 by the generation 𝑃𝑔,𝑡 of that fuel type. This is then divided by national demand 𝐷𝑡 to give the carbon intensity for GB (https://carbonintensity.org.uk/). Here, we are using a dataset with only solar and wind sources.

* The dataset has 7 columns:
- 'from' - a datetime for the 30-minute time window after the quoted datetime

#### GROUND TRUTH
- 'carbon_intensity_actual' - the reported carbon intensity for this 30-minute window in gCO2/kWh

#### FORECASTS
- 'carbon_intensity_forecast' - the forecasted carbon intensity in gCO2/kWh (only to be used for model output comparison)
- 'demand' - the day-ahead (24hr out) demand forecast on the energy grid for this 30-minute window (how much energy is necessary to meet the countries energy needs) in MW (https://www.neso.energy/data-portal/1-day-ahead-demand-forecast)
- 'Solar' - the day-ahead (24hr out) forecast of solar energy production in meeting the demand in this 30-minute window in MW (https://www.neso.energy/data-portal/embedded-wind-and-solar-forecasts/embedded_solar_and_wind_forecast )
- 'Wind Offshore' - the day-ahead (24hr out) forecast for offshore wind energy production in meeting the demand in this 30-minute window in MW
- 'Wind Onshore' - the day-ahead (24hr out) forecast for onshore wind energy production in meeting the demand in this 30-minute window in MW (https://www.neso.energy/data-portal/day-ahead-wind-forecast/day_ahead_wind_forecast)

#### Useful Links
1) https://www.kaggle.com/code/robikscube/tutorial-time-series-forecasting-with-xgboost
2) https://towardsdatascience.com/time-series-for-climate-change-forecasting-energy-demand-79f39c24c85e
3) https://www.sciencedirect.com/science/article/pii/S2772683525000366
4) https://xgboost.readthedocs.io/en/latest/treemethod.html
5) https://carbonintensity.org.uk/
6) https://carbon-intensity.github.io/api-definitions/#intensity
7) https://www.neso.energy/data-portal/national-carbon-intensity-forecast/national_carbon_intensity_forecast_methodology
8) https://www.neso.energy/data-portal/1-day-ahead-demand-forecast
9) https://www.neso.energy/data-portal/day-ahead-wind-forecast/day_ahead_wind_forecast
10) https://github.com/aprabaswara/Forecasting-UK-Electricity-Demand
11) https://seaborn.pydata.org/generated/seaborn.histplot.html
12) https://github.com/OSUKED/CIDataPortal/blob/master/Wrapper%20Development.ipynb
13) https://medium.com/@bhatadithya54764118/day-10-evaluation-metrics-for-regression-mse-mae-rmse-r%C2%B2-score-0ffb39e3ea26
14) https://www.kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning

## Import Packages

In [50]:
import numpy as np
import pandas as pd
import xgboost as xgb
import pickle

## Read CSV File into Pandas

In [51]:
data = pd.read_csv(
    "data/carbon_intensity_demand_solar_wind_2022-01-01_2023-01-01.csv",
    parse_dates=["from"],
    index_col=[0],
)
data.head()

,carbon_intensity_actual,carbon_intensity_forecast,demand,"""Solar""","""Wind Offshore""","""Wind Onshore"""
from,,,,,,
2022-01-01 00:00:00+00:00,74.0,74,21690,0.0,6480.583,4739.050
2022-01-01 00:30:00+00:00,75.0,70,21830,0.0,6480.583,4739.050
2022-01-01 01:00:00+00:00,73.0,70,21335,0.0,6708.648,5380.218
2022-01-01 01:30:00+00:00,68.0,61,20239,0.0,6708.648,5380.218
2022-01-01 02:00:00+00:00,71.0,62,19224,0.0,6978.515,6059.516


XGBoost supports missing values by default. In tree algorithms, branch directions for missing values are learned during training. https://xgboost.readthedocs.io/en/stable/faq.html

# Data Preparation

In our dataframe, we have: datetime, forecasts -> demand, "Solar", "Wind Offshore", "Wind Onshore", and carbon_intensity_actual.
At T=0, forecasting forwards (T+1, T+2, ... T+48), we would have access to past carbon_intensity_actual values, and all of the forecasts (up to 24hr after T=0). 
We will apply some data preprocessing to create additional important features that we can feed the model within this boundary of knowledge at T=0

## Define Functions

These functions are called in the create_features() function. 
* sin_transformer() and cos_transformer() are used to convert temporal features into cyclical features, allowing the model to interpret the hour of the day as a "clock". The sine and cosine functions encode the x,y coordinate of the time on a unit circle, allowing the model to interpret 11.30pm and 00.30am as close together. If they are left as they are, the model interprets these times on opposite ends of a numerical sequence and will see them as "far apart", when in reality they are close together temporally.
* The rest of the functions are to create statistical features that are important for the model to learn trends in the training data. By using the past data and the forecast data, we can compute rolling averages (and its standard deviation, max value, min value), the lag (previous values at T=T-N, where N is the number of steps), the difference (diff) between the current value and a lag value. We can calcluate all of these for multiple different time-steps, feeding the model short/medium/long (hr, day, multi-day) change information and forecasted change

In [52]:
def sin_transformer(x, period):  # function to calculate sine component of cyclical 'unit circles'
    return np.sin(x / period * 2 * np.pi)


def cos_transformer(x, period):
    return np.cos(x / period * 2 * np.pi)  # function to calculate cosine component of cyclical 'unit circles'


def _fmt(
    i,
):  # function to get suffix for data entry names (e.g. 1 timestep = 0.5, 48 timesteps = 24)
    val = i / 2
    return int(val) if val == int(val) else val


def create_roll_mean(
    df, col_name, roll_steps
):  # add columns to df for rolling mean for features UP to T=0 (not into future)
    new_cols = {f"{col_name}_roll_mean_{_fmt(i)}hr": df[col_name].rolling(i).mean() for i in roll_steps if i > 1}
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


def create_roll_mean_std(
    df, col_name, roll_steps
):  # add columns to df for standard deviation of rolling mean for features UP to T=0 (not into future)
    new_cols = {f"{col_name}_roll_mean_std_{_fmt(i)}hr": df[col_name].rolling(i).std() for i in roll_steps if i > 1}
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


def create_roll_max(
    df, col_name, roll_steps
):  # add columns to df for standard deviation of rolling mean for features UP to T=0 (not into future)
    new_cols = {f"{col_name}_roll_max_{_fmt(i)}hr": df[col_name].rolling(i).max() for i in roll_steps if i > 1}
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


def create_roll_min(
    df, col_name, roll_steps
):  # add columns to df for standard deviation of rolling mean for features UP to T=0 (not into future)
    new_cols = {f"{col_name}_roll_min_{_fmt(i)}hr": df[col_name].rolling(i).min() for i in roll_steps if i > 1}
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


def create_lag(
    df, col_name, lag_steps
):  # add columns to df for lag values (e.g. value of feature 2 hours ago) for features UP to T=0 (not into future)
    new_cols = {f"{col_name}_lag_{_fmt(i)}hr": df[col_name].shift(i) for i in lag_steps}
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


def create_diff(
    df, col_name, diff_steps
):  # add columns to df for difference values (e.g. value at T=0 - value at T=-4)) for features UP to T=0 (not into future)
    new_cols = {f"{col_name}_change_{_fmt(i)}hr": df[col_name].diff(i) for i in diff_steps}
    return pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)


def create_roll_lag_diff(
    df, col_name, roll_steps, diff_lag_steps
):  # a function that calls roll, roll_std, diff, lag and returns updated dataframe with new columns
    df = create_roll_mean(df, col_name, roll_steps)
    df = create_roll_max(df, col_name, roll_steps)
    df = create_roll_min(df, col_name, roll_steps)
    df = create_lag(df, col_name, diff_lag_steps)
    df = create_diff(df, col_name, diff_lag_steps)
    df = create_roll_mean_std(df, col_name, roll_steps)
    return df

In [53]:
# fmt: off
# Define the number of steps to use for different features we will calculate and add to the training data
roll_steps = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 18, 20, 22, 24, 32, 36, 48, 336] # Lag, rolling mean, difference steps - these are used for features before T=0 (inform model on trends, changes) 
diff_lag_steps = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 18, 20, 22, 24, 32, 36, 48]
forecast_steps = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16, 18, 20, 22, 24, 32, 36, 48] # Forecast steps - these are used for the demand/solar/wind forecasts as features for model training

DROP_COLS = ['date', 'from', 'carbon_intensity_actual','carbon_intensity_forecast'] # these columns MUST be DROPPED from the training data

In [54]:
def create_features(df):
    df = df.copy()
    df["date"] = df.index
    df["year"] = df["date"].dt.year
    # Time features - convert to individual sine and cosine components for cycilical features
    df["hour_sin"] = sin_transformer(df["date"].dt.hour, 24)  # 24 hours / day, capture daily variations
    df["hour_cos"] = cos_transformer(df["date"].dt.hour, 24)  # 24 hours / day, capture daily variations
    df["dayofweek_sin"] = sin_transformer(df["date"].dt.dayofweek, 7)  # 7 days / week, capture weekly variations
    df["dayofweek_cos"] = cos_transformer(df["date"].dt.dayofweek, 7)  # 7 days / week, capture weekly variations
    df["month_sin"] = sin_transformer(
        df["date"].dt.month, 12
    )  # 12 months / year, capture yearly variations (at monthly granularity)
    df["month_cos"] = cos_transformer(
        df["date"].dt.month, 12
    )  # 12 months / year, capture yearly variations (at monthly granularity)
    df["dayofyear_sin"] = sin_transformer(
        df["date"].dt.dayofyear, 365
    )  # 365 days / year, capture yearly variations (at daily granularity)
    df["dayofyear_cos"] = cos_transformer(
        df["date"].dt.dayofyear, 365
    )  # 365 days / year, capture yearly variations (at daily granularity)

    # One-hot encoded weekdays, weekofyear
    for i, name in enumerate(["dow_mon", "dow_tue", "dow_wed", "dow_thu", "dow_fri", "dow_sat", "dow_sun"]):
        df[name] = (df["date"].dt.dayofweek == i).astype(int)
    for i in range(1, 13):  # One
        df[f"month_{i}"] = (df["date"].dt.month == i).astype(int)

    # 30-min period of day (0-47)
    period = df["date"].dt.hour * 2 + (df["date"].dt.minute >= 30).astype(int)
    df["period_sin"] = sin_transformer(period, 48)
    df["period_cos"] = cos_transformer(period, 48)
    # Identify inputs during peak times of day
    df["is_morning_peak"] = df["date"].dt.hour.between(5, 9).astype(int)  # informed by data exploration
    df["is_evening_peak"] = df["date"].dt.hour.between(17, 21).astype(int)  # informed by data exploration
    # Identify inputs during Christmas and New Years holiday
    df["is_holiday"] = (
        ((df["date"].dt.month == 12) & (df["date"].dt.day >= 24))
        | ((df["date"].dt.month == 1) & (df["date"].dt.day <= 2))
    ).astype(int)
    # Define time-related column names we are adding to the train df
    time_cols = [
        "hour_sin",
        "hour_cos",
        "dayofweek_sin",
        "dayofweek_cos",
        "month_sin",
        "month_cos",
        "dayofyear_sin",
        "dayofyear_cos",
        "dow_mon",
        "dow_tue",
        "dow_wed",
        "dow_thu",
        "dow_fri",
        "dow_sat",
        "dow_sun",
        *[f"month_{i}" for i in range(1, 13)],
        "period_sin",
        "period_cos",
        "year",
    ]

    # Grid features
    df["solar"] = df['"Solar"']  # the 24-hr ahead solar forecast for current T=0
    df["wind_offshore"] = df['"Wind Offshore"']  # the 24-hr ahead offshore wind forecast for current T=0
    df["wind_onshore"] = df['"Wind Onshore"']  # the 24-hr ahead onshore wind forecast for current T=0
    df["residual"] = (
        df["demand"] - df["wind_offshore"] - df["wind_onshore"] - df["solar"]
    )  # the "residual demand" ~ fossil fuels
    df["renewable_total"] = df["wind_offshore"] + df["wind_onshore"] + df["solar"]  # the total renewable contribution
    df["renewable_perc"] = df["renewable_total"] / df["demand"].replace(
        0, np.nan
    )  # the percentage of total demand taken up by renewables

    # Define grid-related column names we are adding to the train df
    grid_cols = [
        "solar",
        "wind_offshore",
        "wind_onshore",
        "residual",
        "renewable_total",
        "renewable_perc",
    ]

    cols_before = set(df.columns)
    # Create lag, rolling mean (+ std), difference features
    df = create_roll_lag_diff(df, "renewable_perc", roll_steps, diff_lag_steps)
    df = create_roll_lag_diff(df, "demand", roll_steps, diff_lag_steps)
    df = create_roll_lag_diff(df, "renewable_total", roll_steps, diff_lag_steps)
    df = create_roll_lag_diff(df, "carbon_intensity_actual", roll_steps, diff_lag_steps)

    df["demand_vs_weekly_mean"] = df["demand"] - df["demand"].rolling(336).mean()
    df["demand_vs_daily_mean"] = df["demand"] - df["demand"].rolling(48).mean()

    lag_roll_diff_cols = list(set(df.columns) - cols_before)
    # Future generation features - iterate through forecast_steps and create features using
    # the forecasts, the only future data we have to inform the model on the state at T+1
    future_cols = {}
    for h in forecast_steps:
        # Shift forecasts backward by h, e.g. [1,2,6]->[0.5hr,1hr,3hr]
        d = df["demand"].shift(-h)
        wo = df['"Wind Offshore"'].shift(-h)
        wn = df['"Wind Onshore"'].shift(-h)
        sol = df['"Solar"'].shift(-h)
        residual = d - wo - wn - sol  # the residual demand (~ from carbon emitting sources)
        future_cols[f"demand_t+{h}"] = d  # future demand (T+1hr) - forecast from previous day
        future_cols[f"wind_off_t+{h}"] = wo  # future offshore wind (T+1hr) - forecast from previous day
        future_cols[f"wind_on_t+{h}"] = wn  # future onshore wind (T+1hr) - forecast from previous day
        future_cols[f"solar_t+{h}"] = sol  # future solar (T+1hr) - forecast from previous day
        future_cols[f"residual_t+{h}"] = residual  # future residual (T+1hr) - forecast from previous day
        future_cols[f"residual_perc_t+{h}"] = residual / d.replace(0, np.nan)
        # # perc. of ~non-renewables meeting the demand
        future_cols[f"residual_delta_t+{h}"] = (
            residual - df["residual"]
        )  # did residual demand go up/down in this time period (0.5hr, 1hr, etc.)
        future_cols[f"ren_perc_t+{h}"] = (wo + wn + sol) / d.replace(
            0, np.nan
        )  # perc. of renewables meeting the demand
        # Perc. of individual renewables meeting the demand
        future_cols[f"solar_perc_t+{h}"] = (sol) / d.replace(0, np.nan)
        future_cols[f"wind_off_perc_t+{h}"] = (wo) / d.replace(0, np.nan)
        future_cols[f"wind_on_perc_t+{h}"] = (wn) / d.replace(0, np.nan)

        future_cols[f"wind_ramp_t+{h}"] = (
            wo - df["wind_offshore"]
        )  # did residual demand go up/down in this time period (0.5hr, 1hr, etc.)
        future_cols[f"ren_ramp_t+{h}"] = (wo + wn + sol) - df[
            "renewable_total"
        ]  # did renewable total go up/down in this time period (0.5hr, 1hr, etc.)
        future_cols[f"solar_ramp_t+{h}"] = (
            sol - df["solar"]
        )  # did solar gen go up/down in this time period (0.5hr, 1hr, etc.)
        future_cols[f"demand_ramp_t+{h}"] = (
            d - df["demand"]
        )  # did demand go up/down in this time period (0.5hr, 1hr, etc.)

    future_feature_cols = list(future_cols.keys())  # get future feature column names
    feature_cols = time_cols + grid_cols + lag_roll_diff_cols + future_feature_cols  # combine all column names

    df = pd.concat([df, pd.DataFrame(future_cols, index=df.index)], axis=1)
    df = df.drop(columns=DROP_COLS, errors="ignore")  # Drop Y,y columns and datetimes
    return df, feature_cols

In [55]:
X, feature_cols = create_features(data)  # run feature creation and transformation of input data

Because of the 168hr (1 week) past rolling mean, the NaN percentage is quite high at 13.36%

In [56]:
X.isna().mean().sort_values(ascending=False) * 100

renewable_total_roll_min_168hr        13.359517
renewable_perc_roll_max_168hr         13.359517
renewable_perc_roll_mean_std_168hr    13.359517
renewable_perc_roll_min_168hr         13.359517
renewable_perc_roll_mean_168hr        13.359517
                                        ...    
month_6                                0.000000
month_5                                0.000000
month_4                                0.000000
month_3                                0.000000
demand                                 0.000000
Length: 864, dtype: float64

In [57]:
# Build 48-column target matrix — one column per forecast horizon
Y = pd.concat(
    {f"ci_t+{h}": data["carbon_intensity_actual"].shift(-h) for h in range(1, 49)},
    axis=1,
)

# Drop rows where any feature or any target is NaN
df_combined = pd.concat([X, Y], axis=1).dropna()
X_clean = df_combined[X.columns]
Y_clean = df_combined[Y.columns]

# Chronological 80/20 split — no leakage risk
split = int(len(X_clean) * 0.80)
X_train, X_test = X_clean.iloc[:split], X_clean.iloc[split:]
Y_train, Y_test = Y_clean.iloc[:split], Y_clean.iloc[split:]

FEATURE_COLS = list(X_train.columns)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Test period: {X_test.index.min().date()} -> {X_test.index.max().date()}")

Train: (11708, 864), Test: (2927, 864)
Test period: 2022-10-05 -> 2022-12-31


In [ ]:
# Multi-output model using XGBoost - predict 24-hr forecast as one output
reg = xgb.XGBRegressor(
    n_estimators=3000,  # the number of trees (boosting rounds). A high number requires early stopping to prevent overfitting.
    learning_rate=0.03,  # determines how quickly the model learns, with smaller values reducing the risk of overfitting
    max_depth=5,  # maximum depth of a tree. Increasing this value will make the model more complex and more likely to overfit.
    min_child_weight=3,  # minimum sum of instance weight (hessian) needed in a child.
    subsample=0.8,  # fraction of training data to sample for each tree
    colsample_bytree=0.6,  # fraction of features to sample for each tree
    reg_lambda=1.0,  # L2 regularisation term on weights. Increasing this value will make model more conservative.
    reg_alpha=0.0,  # L1 regularisation term on weights. Increasing this value will make model more conservative.
    gamma=0.0,  # minimum loss reduction required to make a further partition on a leaf node of the tree. The larger gamma is, the more conservative the algorithm will be.
    tree_method="hist",  # use tree_method='hist' for faster training on large datasets.
    early_stopping_rounds=100,  # early stopping to stop over-fitting (this could be lower looking at the train-test rmse's)
)

# validation_0 = training rmse, validation_1 = test_rmse
reg.fit(
    X_train,
    Y_train,
    eval_set=[(X_train, Y_train), (X_test, Y_test)],
    verbose=100,
)

[0]	validation_0-rmse:59.81984	validation_1-rmse:64.10316
[100]	validation_0-rmse:12.08822	validation_1-rmse:22.79905
[200]	validation_0-rmse:8.29960	validation_1-rmse:22.55840
[300]	validation_0-rmse:7.00685	validation_1-rmse:22.52521
[400]	validation_0-rmse:6.12649	validation_1-rmse:22.50517
[500]	validation_0-rmse:5.43700	validation_1-rmse:22.49632
[600]	validation_0-rmse:4.87775	validation_1-rmse:22.49310
[700]	validation_0-rmse:4.41467	validation_1-rmse:22.49680
[723]	validation_0-rmse:4.31933	validation_1-rmse:22.49673


,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6
,device,None
,early_stopping_rounds,100
,enable_categorical,False
,eval_metric,None


## Save Model to File

In [59]:
# Source - https://stackoverflow.com/a/57874903
# Posted by ChrisDanger, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-17, License - CC BY-SA 4.0

OUTPUT_DIR = "data/outputs/model"

# Save model
pickle.dump(reg, open(f"{OUTPUT_DIR}/xgb_regressor_carbon_intensity_forecast.pkl", "wb"))

# Save datasets
for name, obj in [
    ("X_train", X_train),
    ("X_test", X_test),
    ("Y_train", Y_train),
    ("Y_test", Y_test),
]:
    pickle.dump(obj, open(f"{OUTPUT_DIR}/{name}.pkl", "wb"))

print("Saved model + datasets to", OUTPUT_DIR)

Saved model + datasets to data/outputs/model
